# 🧠 Random Forests Explanation and Hands-on Example

Welcome to the hands-on explanation notebook for **Random Forests**! In this notebook, we will:
1. Generate a complex, noisy binary classification dataset.
2. Train a single **Decision Tree** vs. a **Random Forest** using `scikit-learn` to observe how ensembling smooths decision boundaries.
3. Implement a **Random Forest from scratch** by building an ensemble of bootstrap-sampled decision trees with random feature selection.
4. Evaluate our scratch implementation and compare its performance with scikit-learn.
5. Explain how ensembling concepts relate to deep learning (e.g., Model Averaging, Test Time Augmentation).

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from collections import Counter

# Set seed for reproducibility
np.random.seed(42)

## 1. Data Generation

We will generate a noisy "Double Moons" dataset, which has non-linear and overlapping boundaries, representing the messy feature spaces often found in real-world computer vision tasks.

In [ ]:
# Generate noisy double moons dataset
X_train, y_train = make_moons(n_samples=150, noise=0.35, random_state=42)
X_test, y_test = make_moons(n_samples=100, noise=0.35, random_state=42)

# Plot the dataset
plt.figure(figsize=(8, 5))
plt.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], color='red', label='Class 0: Handwheel Valve', alpha=0.7)
plt.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], color='blue', label='Class 1: Lever Valve', alpha=0.7)
plt.xlabel('Visual Feature 1')
plt.ylabel('Visual Feature 2')
plt.title('PTT Component Classification Dataset')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

## 2. Decision Tree vs. Random Forest Boundaries

Let's visualize the decision boundaries of a single Decision Tree vs. Random Forests with varying number of trees.

In [ ]:
# Generate grid points for boundaries
x_min, x_max = X_train[:, 0].min() - 0.5, X_train[:, 0].max() + 0.5
y_min, y_max = X_train[:, 1].min() - 0.5, X_train[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))
grid_points = np.c_[xx.ravel(), yy.ravel()]

# Models to compare
models = {
    'Single Decision Tree': DecisionTreeClassifier(max_depth=None, random_state=42),
    'Random Forest (10 Trees)': RandomForestClassifier(n_estimators=10, random_state=42),
    'Random Forest (100 Trees)': RandomForestClassifier(n_estimators=100, random_state=42)
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cmap_light = ListedColormap(['#FFAAAA', '#AAAAFF'])
cmap_bold = ['red', 'blue']

for idx, (name, clf) in enumerate(models.items()):
    clf.fit(X_train, y_train)
    Z = clf.predict(grid_points)
    Z = Z.reshape(xx.shape)
    
    # Evaluate test accuracy
    test_acc = accuracy_score(y_test, clf.predict(X_test))
    
    ax = axes[idx]
    ax.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.6)
    ax.scatter(X_train[:, 0], X_train[:, 1], c=[cmap_bold[int(i)] for i in y_train], edgecolor='k', s=35)
    ax.set_title(f'{name}\nTest Accuracy: {test_acc * 100:.1f}%')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

*   **Single Decision Tree:** The boundary is highly complex, jagged, and contains isolated "islands" of classes, overfitting to noise.
*   **Random Forest (100 Trees):** The boundary is much smoother and models the underlying double moon shape much better, resulting in significantly higher test accuracy.

## 3. Random Forest from Scratch

Let's build a Custom Random Forest. For each tree:
1.  Generate a **Bootstrap Sample**: Randomly choose $m$ training samples with replacement.
2.  Select a **Random Subset of Features**: Select features randomly for each tree.
3.  Train a standard Decision Tree.
4.  Aggregates predictions across all trees using majority voting.

In [ ]:
class CustomRandomForest:
    def __init__(self, n_estimators=10, max_depth=3, max_features=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.trees = []
        self.feat_indices = []

    def _bootstrap_samples(self, X, y):
        m = X.shape[0]
        indices = np.random.choice(m, m, replace=True)
        return X[indices], y[indices]

    def fit(self, X, y):
        self.trees = []
        self.feat_indices = []
        m, n = X.shape
        
        # Determine number of features to sample
        if self.max_features is None:
            n_features_to_sample = n
        elif self.max_features == 'sqrt':
            n_features_to_sample = int(np.sqrt(n))
        else:
            n_features_to_sample = self.max_features
            
        for _ in range(self.n_estimators):
            # 1. Generate bootstrap dataset
            X_boot, y_boot = self._bootstrap_samples(X, y)
            
            # 2. Select random subset of feature indexes
            feat_idx = np.random.choice(n, n_features_to_sample, replace=False)
            self.feat_indices.append(feat_idx)
            
            # 3. Fit decision tree on subset of data
            tree = DecisionTreeClassifier(max_depth=self.max_depth)
            tree.fit(X_boot[:, feat_idx], y_boot)
            self.trees.append(tree)

    def predict(self, X):
        tree_preds = []
        for tree, feat_idx in zip(self.trees, self.feat_indices):
            preds = tree.predict(X[:, feat_idx])
            tree_preds.append(preds)
            
        tree_preds = np.array(tree_preds).T
        
        # Majority vote
        final_preds = np.array([Counter(row).most_common(1)[0][0] for row in tree_preds])
        return final_preds

# Fit our custom scratch Random Forest
forest_scratch = CustomRandomForest(n_estimators=100, max_depth=5, max_features=2)
forest_scratch.fit(X_train, y_train)

# Evaluate Custom RF
y_pred_scratch = forest_scratch.predict(X_test)
scratch_acc = accuracy_score(y_test, y_pred_scratch)

print(f"Scratch Random Forest Test Accuracy: {scratch_acc * 100:.2f}%")

## 4. Visualizing Custom Random Forest Boundaries

Let's plot the decision boundaries of our Custom Scratch Random Forest.

In [ ]:
Z_scratch = forest_scratch.predict(grid_points)
Z_scratch = Z_scratch.reshape(xx.shape)

plt.figure(figsize=(8, 5))
plt.contourf(xx, yy, Z_scratch, cmap=cmap_light, alpha=0.6)
plt.scatter(X_train[:, 0], X_train[:, 1], c=[cmap_bold[int(i)] for i in y_train], edgecolor='k', s=40)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title(f'Scratch Random Forest Boundary (100 Trees)\nTest Accuracy: {scratch_acc * 100:.1f}%')
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

## 💡 Connection to Deep Learning & YOLO
*   **Ensemble Averaging:** Training multiple instances of neural networks (e.g. 5 YOLO models trained with different seeds) and averaging their predicted bounding boxes is a standard approach to win kaggle competitions or maximize production safety.
*   **Test Time Augmentation (TTA):** During inference, we run a single YOLO model on multiple augmented versions of the input image (horizontal flip, zoomed, color shifts) and aggregate the predictions. This creates a virtual ensemble, smoothing predictions in the same way bagging does.